# Notebook 3 — Методы матричной факторизации
## CTKT и GiPM3F (адаптированные постановки)

**Цель ноутбука:**  
Реализовать и оценить два метода класса матричной факторизации,
адаптированных для детекции мошеннических транзакций на датасете Sparkov.

**Методы:**
- **CTKT** (Collaborative Transaction ranKing and raTing) — Cui et al., 2021.  
  Прямая трансформация задачи фрода в псевдо-рекомендательную через
  частоты совместных появлений поведенческих атрибутов.
- **GiPM3F** (Gibbs iPM3F) — Yang et al., 2018.  
  Матричная факторизация типа max-margin, ошибка предсказания которой
  служит скоринговой функцией аномальности.

**Входные данные:**  
Оба метода работают с **сырыми** транзакциями (`data/raw/`), а не с
предобработанными признаками из Notebook 1. Это методологически корректно:
каждый метод сам конструирует своё представление данных согласно
оригинальной архитектуре. Feature-engineered признаки из Notebook 1
предназначены только для baseline ML-моделей.

**Адаптации относительно оригинальных статей:**  
Подробно документируются в соответствующих разделах ноутбука.
Все отступления от оригинала обоснованы спецификой датасета Sparkov
и не затрагивают ключевые архитектурные решения методов.

**Выходные данные:**  
Скоры обеих моделей сохраняются в `results/` для финального
сравнения в Notebook 6.

## 1. Импорты и загрузка данных

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import json
import warnings
from pathlib import Path
from itertools import product
from collections import defaultdict

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score,
    ndcg_score, precision_recall_curve, roc_curve,
)

warnings.filterwarnings("ignore")

# Пути
RAW       = Path("../data/raw")
PROCESSED = Path("../data/processed")
RESULTS   = Path("../results")
MODELS    = Path("../models")

# Устройство: MPS на M1, иначе CPU
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")
print(f"Устройство: {DEVICE}")

# Стиль
plt.rcParams["figure.dpi"]  = 120
plt.rcParams["font.family"] = "DejaVu Sans"
sns.set_style("whitegrid")
PALETTE = {"ctkt": "#2563EB", "gipm3f": "#DC2626", "baseline": "#6B7280"}

# Воспроизводимость
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

Устройство: mps


## 2. Загрузка и подготовка сырых данных

Загружаем сырые транзакции и выполняем минимальную предобработку,
необходимую для обоих методов:
- Парсинг дат
- Дискретизация непрерывных атрибутов (сумма, час)
- Построение словарей идентификаторов

**Важно:** никакого feature engineering из Notebook 1 здесь не применяется.
Каждый метод далее строит своё представление данных самостоятельно.

In [2]:
# Загрузка
train_raw = pd.read_csv(RAW / "fraudTrain.csv", index_col=0)
test_raw  = pd.read_csv(RAW / "fraudTest.csv",  index_col=0)

print(f"Train: {train_raw.shape[0]:,} строк")
print(f"Test:  {test_raw.shape[0]:,} строк")
print(f"Доля фрода train: {train_raw['is_fraud'].mean():.4%}")
print(f"Доля фрода test:  {test_raw['is_fraud'].mean():.4%}")

Train: 1,296,675 строк
Test:  555,719 строк
Доля фрода train: 0.5789%
Доля фрода test:  0.3860%


In [3]:
# Парсинг дат
for df in [train_raw, test_raw]:
    dt = pd.to_datetime(df["trans_date_trans_time"])
    df["hour"]        = dt.dt.hour
    df["day_of_week"] = dt.dt.dayofweek
    df["month"]       = dt.dt.month

# Дискретизация суммы на 10 бинов (как в оригинальной статье CTKT)
# Авторы дискретизируют amount на 10 значений по квантилям train
amt_bins = pd.qcut(train_raw["amt"], q=10, labels=False, duplicates="drop")
amt_quantiles = train_raw["amt"].quantile(
    [i/10 for i in range(11)]
).values

for df in [train_raw, test_raw]:
    df["amt_bin"] = pd.cut(
        df["amt"],
        bins=amt_quantiles,
        labels=False,
        include_lowest=True
    ).fillna(0).astype(int)

# Дискретизация часа на 4 части суток (как в оригинальной статье CTKT)
# Авторы делят день на 4 части × 2 типа дня (будни/выходные) = 8 значений
def hour_to_period(hour, is_weekend):
    period = hour // 6  # 0-5: ночь, 6-11: утро, 12-17: день, 18-23: вечер
    return period + (4 if is_weekend else 0)

for df in [train_raw, test_raw]:
    is_weekend = (df["day_of_week"] >= 5).astype(int)
    df["time_period"] = df.apply(
        lambda r: hour_to_period(r["hour"], r["day_of_week"] >= 5), axis=1
    )

print("\nДискретизация выполнена")
print(f"Уникальных amt_bin: {train_raw['amt_bin'].nunique()}")
print(f"Уникальных time_period: {train_raw['time_period'].nunique()}")


Дискретизация выполнена
Уникальных amt_bin: 10
Уникальных time_period: 8


## 3. CTKT — Collaborative Transaction ranKing and raTing

### 3.1 Архитектура и ключевые идеи

CTKT трансформирует задачу детекции фрода в псевдо-рекомендательную.
Логика преобразования:

| Рекомендательная система | CTKT (антифрод) |
|---|---|
| Пользователь | Контекстуальный атрибут (держатель карты, мерчант, штат) |
| Товар | Прототипическая транзакция |
| Рейтинг | Псевдо-рейтинг из частот совместных появлений |

**Прототипическая транзакция** — комбинация дискретизированных
поведенческих атрибутов. Два события с одинаковыми значениями всех
поведенческих атрибутов считаются одним и тем же прототипом.

**Псевдо-рейтинг** отражает степень легитимности пары
$(\text{контекст},\ \text{прототип})$ через частоты появлений с разными метками:

$$r = \sigma(N^-) - \sigma(N^+) + 0.5$$

где $N^-$ — частота появления пары с легитимной меткой ($\texttt{is\_fraud}=0$),
$N^+$ — с мошеннической ($\texttt{is\_fraud}=1$), $\sigma(x) = \frac{1}{1+e^{-x}}$.
Рейтинг $r > 0.5$ соответствует преимущественно легитимному поведению,
$r < 0.5$ — мошенническому. Чем больше накоплено наблюдений,
тем выше уверенность в метке.

**Функция потерь** объединяет два критерия в экспоненциальной форме (eq. 6):

$$\mathcal{T} = \prod_{C \in \mathcal{A}_c} \prod_{\omega \in \Omega^C}
\Pr(\omega)^{\exp\!\left(-\gamma \sum_{e \in \{e_1, e_2\}}
\bigl(r(u_C, e) - \sigma(f_{u_C}(e))\bigr)^2\right)}$$

где $0 < \gamma < 1$ контролирует вклад рейтинговой части.

В логарифмической форме (eq. 8) для численной стабильности:

$$\mathcal{O} = -\log \mathcal{T} + \frac{\lambda}{2}\|\Theta\|_F^2$$

**Ранжировочная часть** (eq. 4):

$$\Pr(\omega) = \sigma\!\bigl(f_{u_C}(e_1) - f_{u_C}(e_2)
- \varepsilon \cdot f_{u_C}(e_1) \cdot f_{u_C}(e_2)\bigr)$$

где $\varepsilon$ усиливает разрыв между скорами легитимного и
мошеннического прототипов.

**Рейтинговая часть** (eq. 5):

$$\mathcal{T}_{\text{rating}} = \sum_{C \in \mathcal{A}_c}
\sum_{e \in E} \bigl(r_{u_C, e} - \sigma(f_{u_C}(e))\bigr)^2$$

### 3.2 Адаптация к датасету Sparkov

Оригинальная статья использует следующие атрибуты:
- Контекстуальные ($\to$ псевдо-пользователи): account, merchant, place
- Поведенческие ($\to$ прототип): time, amount, recency, os, ip

Адаптация под Sparkov:

| Оригинал | Sparkov | Тип | Обоснование |
|---|---|---|---|
| account | cc_num | Контекстуальный | Прямое соответствие: идентификатор карты/аккаунта |
| merchant | merchant | Контекстуальный | Прямое соответствие |
| place | state | Контекстуальный | Географический контекст транзакции |
| time | time_period | Поведенческий | Час дня × тип дня (8 значений, как в статье) |
| amount | amt_bin | Поведенческий | Дискретизация на 10 квантилей по train |
| recency | day_of_week | Поведенческий | Временной паттерн активности |
| os | category | Поведенческий | Тип операции как поведенческий контекст |
| ip | gender | Поведенческий | Демографический контекст держателя |

**Что сохранено из оригинала без изменений:**
формула псевдо-рейтинга (eq. 1), экспоненциальная комбинация
критериев (eq. 6), параметризация через эмбеддинги (eq. 7),
SGD с бутстрэпингом по preference events (Algorithm 1).

**Что адаптировано:**
- `cc_num` используется из сырых данных — в Notebook 1 он был дропнут
  как идентификатор для ML-моделей, но здесь выполняет роль `account`
  для группировки транзакций, что концептуально верно
- `os` и `ip` отсутствуют в Sparkov, заменены на `category` и `gender`
  как ближайшие поведенческие аналоги

### 3.3 Построение псевдо-рекомендательной системы

Конвейер построения:

1. Строим словари кодирования для всех атрибутов по train
2. Для каждой пары $(u_C,\ e)$ считаем частоты $N^-$ и $N^+$ по train
3. Вычисляем псевдо-рейтинги: $r = \sigma(N^-) - \sigma(N^+) + 0.5$
4. Формируем preference events $e_p \succ_{u_C} e_q$ для обучения
   по Algorithm 1

Все статистики считаются **только по train** — утечки на test нет.

In [4]:
CONTEXTUAL_ATTRS = ["cc_num", "merchant", "state"]
BEHAVIORAL_ATTRS = ["time_period", "amt_bin", "day_of_week", "category", "gender"]

def build_vocab(df, cols):
    """
    Строит словарь {значение: int_id} для каждой колонки.
    Только по train — test-значения которых нет в словаре
    будут пропускаться при inference.
    """
    vocabs = {}
    for col in cols:
        unique_vals = df[col].astype(str).unique()
        vocabs[col] = {v: i for i, v in enumerate(sorted(unique_vals))}
    return vocabs

# Словари строятся только по train
ctx_vocabs = build_vocab(train_raw, CONTEXTUAL_ATTRS)
beh_vocabs = build_vocab(train_raw, BEHAVIORAL_ATTRS)

print("Размеры словарей контекстуальных атрибутов:")
for col in CONTEXTUAL_ATTRS:
    print(f"  {col}: {len(ctx_vocabs[col]):,} уникальных значений")

print("\nРазмеры словарей поведенческих атрибутов:")
for col in BEHAVIORAL_ATTRS:
    print(f"  {col}: {len(beh_vocabs[col])} уникальных значений")

Размеры словарей контекстуальных атрибутов:
  cc_num: 983 уникальных значений
  merchant: 693 уникальных значений
  state: 51 уникальных значений

Размеры словарей поведенческих атрибутов:
  time_period: 8 уникальных значений
  amt_bin: 10 уникальных значений
  day_of_week: 7 уникальных значений
  category: 14 уникальных значений
  gender: 2 уникальных значений


In [5]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -500, 500)))

def build_cooccurrence(df, ctx_attr, beh_attrs, ctx_vocab, beh_vocabs):
    """
    Для каждой пары (pseudo_user, prototype) считает:
        N⁻ — число появлений с is_fraud=0
        N⁺ — число появлений с is_fraud=1

    Псевдо-рейтинг: r = σ(N⁻) - σ(N⁺) + 0.5  (eq. 1, Cui et al. 2021)
    r > 0.5 - легитимное поведение
    r < 0.5 - мошенническое поведение

    Все статистики считаются только по переданному df (train).
    """
    cooc = defaultdict(lambda: {"N_neg": 0, "N_pos": 0})

    for _, row in df.iterrows():
        u_val = str(row[ctx_attr])
        if u_val not in ctx_vocab:
            continue
        u_id = ctx_vocab[u_val]

        # Прототип — кортеж закодированных поведенческих значений
        proto = tuple(
            beh_vocabs[col].get(str(row[col]), -1)
            for col in beh_attrs
        )
        if -1 in proto:
            continue

        if int(row["is_fraud"]) == 0:
            cooc[(u_id, proto)]["N_neg"] += 1
        else:
            cooc[(u_id, proto)]["N_pos"] += 1

    # Вычисляем псевдо-рейтинги
    result = {}
    for (u_id, proto), counts in cooc.items():
        N_neg  = counts["N_neg"]
        N_pos  = counts["N_pos"]
        rating = sigmoid(N_neg) - sigmoid(N_pos) + 0.5
        result[(u_id, proto)] = {
            "N_neg":  N_neg,
            "N_pos":  N_pos,
            "rating": float(np.clip(rating, 0.0, 1.0)),
        }

    return result

print("Строим матрицы совместных появлений...")

cooc_data = {}
for ctx_attr in CONTEXTUAL_ATTRS:
    t0 = time.time()
    cooc_data[ctx_attr] = build_cooccurrence(
        train_raw, ctx_attr, BEHAVIORAL_ATTRS,
        ctx_vocabs[ctx_attr], beh_vocabs
    )
    n_pairs = len(cooc_data[ctx_attr])
    n_fraud = sum(
        1 for v in cooc_data[ctx_attr].values()
        if v["rating"] < 0.5
    )
    print(f"  {ctx_attr}:")
    print(f"    Уникальных пар:          {n_pairs:,}")
    print(f"    Мошеннических (r < 0.5): {n_fraud:,} ({n_fraud/n_pairs:.2%})")
    print(f"    Время:                   {time.time()-t0:.1f}s\n")

Строим матрицы совместных появлений...
  cc_num:
    Уникальных пар:          638,113
    Мошеннических (r < 0.5): 4,766 (0.75%)
    Время:                   26.9s

  merchant:
    Уникальных пар:          212,202
    Мошеннических (r < 0.5): 1,351 (0.64%)
    Время:                   23.7s

  state:
    Уникальных пар:          178,273
    Мошеннических (r < 0.5): 1,297 (0.73%)
    Время:                   23.6s



### 3.4 Реализация модели CTKT

Параметризация через эмбеддинги (eq. 7):

- Каждый псевдо-пользователь $u_C$ ассоциируется с матрицей предпочтений
  $M_{u_C} \in \mathbb{R}^{d \times J}$, где $d$ — размерность эмбеддинга,
  $J$ — число поведенческих атрибутов
- Значение $v_{j,k_j}$ $j$-го поведенческого атрибута ассоциируется
  с $d$-мерным вектором эмбеддинга
- Скор предпочтения:

$$f_{u_C}(e) = \sum_{j=1}^{J} g\!\left(M_{u_C,j},\ v_{j,k_j}\right)
= \sum_{j=1}^{J} M_{u_C,j}^{\top} v_{j,k_j}$$

где $g(x, y) = x \cdot y$ — скалярное произведение соответствующих
столбцов матриц $M_{u_C}$ и $A_e$.

In [6]:
class CTKTModel(nn.Module):
    """
    Реализация CTKT (Cui et al., 2021).

    Параметризация через эмбеддинги (eq. 7):
    - user_embeddings: матрицы предпочтений M_u ∈ R^(d×J)
    - item_embeddings: векторы значений v_j,k для каждого
      поведенческого атрибута

    Скор: f(u, e) = Σⱼ dot(M_u[:,j], v_j,k_j)
    """

    def __init__(
        self,
        n_users:      int,
        n_beh_sizes:  list,   # число уникальных значений каждого бех. атрибута
        emb_dim:      int = 16,
    ):
        super().__init__()
        self.J       = len(n_beh_sizes)
        self.emb_dim = emb_dim

        # Матрица предпочтений псевдо-пользователей: (n_users, d*J)
        # Разворачиваем d×J в вектор для удобства индексации
        self.user_emb = nn.Embedding(n_users, emb_dim * self.J)

        # Эмбеддинги значений поведенческих атрибутов
        # Для j-го атрибута: таблица (n_vals_j, d)
        self.item_embs = nn.ModuleList([
            nn.Embedding(n_vals + 1, emb_dim, padding_idx=n_vals)
            for n_vals in n_beh_sizes
        ])

        # Инициализация по нормальному распределению (Algorithm 1)
        nn.init.normal_(self.user_emb.weight, mean=0.0, std=0.01)
        for emb in self.item_embs:
            nn.init.normal_(emb.weight, mean=0.0, std=0.01)

    def score(self, user_ids: torch.Tensor,
              proto_ids: torch.Tensor) -> torch.Tensor:
        """
        Вычисляет скор f(u, e) для батча пар (user, prototype).

        user_ids:  (B,)
        proto_ids: (B, J) — закодированные значения J атрибутов прототипа

        Возвращает: (B,) — скоры
        """
        # user_emb: (B, d*J) - reshape (B, J, d)
        u = self.user_emb(user_ids).view(-1, self.J, self.emb_dim)

        # Для каждого атрибута j получаем эмбеддинг значения: (B, d)
        # Скор = Σⱼ dot(u[:,j,:], v_j)
        s = torch.zeros(user_ids.size(0), device=user_ids.device)
        for j, emb_layer in enumerate(self.item_embs):
            v_j = emb_layer(proto_ids[:, j])   # (B, d)
            s   = s + (u[:, j, :] * v_j).sum(dim=1)  # dot product

        return s

    def forward(self, user_ids, proto_pos, proto_neg, r_pos, r_neg, eps, gamma):
        """
        Вычисляет функцию потерь CTKT (eq. 6 / eq. 8).

        Ранжировочная часть (eq. 3–4):
            Pr(ω) = σ(f(u,e+) - f(u,e-) - ε·f(u,e+)·f(u,e-))

        Рейтинговая часть (eq. 5):
            L_rating = (r_pos - σ(f_pos))² + (r_neg - σ(f_neg))²

        Объединение (eq. 6 в лог-форме eq. 8):
            loss = -log Pr(ω) + γ · L_rating
        """
        f_pos = self.score(user_ids, proto_pos)   # (B,)
        f_neg = self.score(user_ids, proto_neg)   # (B,)

        # Ранжировочная часть
        diff    = f_pos - f_neg - eps * f_pos * f_neg
        pr_omega = torch.sigmoid(diff)
        rank_loss = -torch.log(pr_omega + 1e-9).mean()

        # Рейтинговая часть
        sig_pos  = torch.sigmoid(f_pos)
        sig_neg  = torch.sigmoid(f_neg)
        rate_loss = (
            (r_pos - sig_pos).pow(2) + (r_neg - sig_neg).pow(2)
        ).mean()

        loss = rank_loss + gamma * rate_loss
        return loss

### 3.5 Формирование preference events и обучение

Согласно Algorithm 1 (Cui et al. 2021), обучение происходит
через бутстрэпинг preference events:

$$e_p \succ_{u_C} e_q \quad \Leftrightarrow \quad
r_p > r_q,\ r_p > 0.5,\ r_q \leq 0.5$$

Условия для preference event (eq. 2):
- **Условие 1** ($\Omega^C_1$): $r_p > r_q,\ r_p > 0.5,\ r_q < 0.5$
  — один легитимный, другой мошеннический прототип; $\varepsilon$ больше
- **Условие 2** ($\Omega^C_2$): $r_p > r_q,\ r_p > 0.5,\ r_q = 0.5$
  — один легитимный, другой неопределённый; $\varepsilon$ меньше

В каждой итерации случайно сэмплируем контекст $C \in \mathcal{A}_c$,
затем случайную preference event из $\Omega^C = \Omega^C_1 \cup \Omega^C_2$.

In [7]:
class CTKTDataset(Dataset):
    """
    Формирует preference events для обучения CTKT.

    Для каждого контекстуального атрибута и каждого псевдо-пользователя
    находим все пары (легитимный прототип, мошеннический прототип)
    и сохраняем их как preference events.
    """

    def __init__(self, cooc_data: dict, ctx_attr: str):
        self.events = []

        # Группируем пары по псевдо-пользователю
        user_to_pairs = defaultdict(lambda: {"pos": [], "neg": []})
        for (u_id, proto), info in cooc_data[ctx_attr].items():
            r = info["rating"]
            if r > 0.5:
                user_to_pairs[u_id]["pos"].append((proto, r))
            elif r < 0.5:
                user_to_pairs[u_id]["neg"].append((proto, r))

        # Формируем preference events: (u, e+, e-, r+, r-)
        for u_id, pairs in user_to_pairs.items():
            pos_list = pairs["pos"]
            neg_list = pairs["neg"]
            if not pos_list or not neg_list:
                continue
            for (proto_p, r_p) in pos_list:
                for (proto_q, r_q) in neg_list:
                    self.events.append((
                        u_id,
                        proto_p, proto_q,
                        r_p, r_q
                    ))

        print(f"  Preference events для {ctx_attr}: {len(self.events):,}")

    def __len__(self):
        return len(self.events)

    def __getitem__(self, idx):
        u_id, proto_p, proto_q, r_p, r_q = self.events[idx]
        return (
            torch.tensor(u_id,    dtype=torch.long),
            torch.tensor(proto_p, dtype=torch.long),
            torch.tensor(proto_q, dtype=torch.long),
            torch.tensor(r_p,     dtype=torch.float32),
            torch.tensor(r_q,     dtype=torch.float32),
        )

In [8]:
def train_ctkt(
    cooc_data:   dict,
    ctx_attr:    str,
    ctx_vocabs:  dict,
    beh_vocabs:  dict,
    beh_attrs:   list,
    emb_dim:     int   = 16,
    n_epochs:    int   = 5,
    batch_size:  int   = 2048,
    lr:          float = 0.01,
    lambda_reg:  float = 0.001,
    eps:         float = 0.1,
    gamma:       float = 0.5,
) -> CTKTModel:
    """
    Обучает CTKT для одного контекстуального атрибута.

    Гиперпараметры:
        emb_dim    — размерность латентного пространства (d в статье)
        eps        — параметр усиления разрыва между скорами (eq. 4)
        gamma      — вес рейтинговой части в комбинированной потере (eq. 6)
        lambda_reg — коэффициент L2-регуляризации (eq. 8)
    """
    n_users     = len(ctx_vocabs[ctx_attr])
    n_beh_sizes = [len(beh_vocabs[col]) for col in beh_attrs]

    dataset    = CTKTDataset(cooc_data, ctx_attr)
    dataloader = DataLoader(
        dataset, batch_size=batch_size, shuffle=True, num_workers=0
    )

    model = CTKTModel(n_users, n_beh_sizes, emb_dim).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=lr,
                           weight_decay=lambda_reg)

    model.train()
    for epoch in range(n_epochs):
        total_loss = 0.0
        n_batches  = 0

        for batch in dataloader:
            u_ids, proto_p, proto_q, r_p, r_q = [
                b.to(DEVICE) for b in batch
            ]

            optimizer.zero_grad()
            loss = model(u_ids, proto_p, proto_q, r_p, r_q,
                         eps=eps, gamma=gamma)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            n_batches  += 1

        avg_loss = total_loss / max(n_batches, 1)
        print(f"    Epoch {epoch+1}/{n_epochs}  loss={avg_loss:.4f}")

    return model

print("Обучаем CTKT для каждого контекстуального атрибута...\n")

ctkt_models = {}
for ctx_attr in CONTEXTUAL_ATTRS:
    print(f"── {ctx_attr} ──")
    t0 = time.time()
    ctkt_models[ctx_attr] = train_ctkt(
        cooc_data  = cooc_data,
        ctx_attr   = ctx_attr,
        ctx_vocabs = ctx_vocabs,
        beh_vocabs = beh_vocabs,
        beh_attrs  = BEHAVIORAL_ATTRS,
    )
    print(f"  Время обучения: {time.time()-t0:.1f}s\n")

Обучаем CTKT для каждого контекстуального атрибута...

── cc_num ──
  Preference events для cc_num: 2,761,311
    Epoch 1/5  loss=0.1764
    Epoch 2/5  loss=0.1650
    Epoch 3/5  loss=0.1637
    Epoch 4/5  loss=0.1628
    Epoch 5/5  loss=0.1624
  Время обучения: 197.6s

── merchant ──
  Preference events для merchant: 366,617
    Epoch 1/5  loss=0.1889
    Epoch 2/5  loss=0.1040
    Epoch 3/5  loss=0.0996
    Epoch 4/5  loss=0.0992
    Epoch 5/5  loss=0.0989
  Время обучения: 24.8s

── state ──
  Preference events для state: 4,956,660
    Epoch 1/5  loss=0.1089
    Epoch 2/5  loss=0.1025
    Epoch 3/5  loss=0.1022
    Epoch 4/5  loss=0.1021
    Epoch 5/5  loss=0.1020
  Время обучения: 361.0s



### 3.6 Инференс и скоринг

Для каждой транзакции из test вычисляем скор аномальности.

Логика скоринга:
- Определяем прототип транзакции по поведенческим атрибутам
- Для каждого контекстуального атрибута $C$ получаем скор
  $f_{u_C}(e)$ из соответствующей модели
- Итоговый скор — среднее по трём контекстам:

$$s(u, m, x) = \frac{1}{|\mathcal{A}_c|}
\sum_{C \in \mathcal{A}_c} \left(1 - \sigma\!\left(f_{u_C}(e)\right)\right)$$

Инверсия $(1 - \sigma(\cdot))$ необходима потому что высокий скор
модели соответствует **легитимному** поведению, тогда как для
детекции фрода нам нужен высокий скор для **аномального**.

Транзакции из test, чьи контекстуальные атрибуты не встречались
в train (unseen users/merchants), получают нейтральный скор 0.5.

In [9]:
def score_transactions(
    df:          pd.DataFrame,
    cooc_data:   dict,
    ctkt_models: dict,
    ctx_vocabs:  dict,
    beh_vocabs:  dict,
    ctx_attrs:   list,
    beh_attrs:   list,
) -> np.ndarray:
    """
    Вычисляет скор аномальности для каждой транзакции в df.

    Скор = среднее по контекстуальным атрибутам из (1 - σ(f(u,e))):
    высокий скор - транзакция аномальна для данного контекста.

    Unseen контексты или прототипы получают нейтральный скор 0.5.
    """
    n = len(df)
    scores_per_ctx = np.full((n, len(ctx_attrs)), 0.5)

    for ctx_idx, ctx_attr in enumerate(ctx_attrs):
        model = ctkt_models[ctx_attr]
        model.eval()

        ctx_vocab = ctx_vocabs[ctx_attr]
        u_ids_list    = []
        proto_list    = []
        valid_indices = []

        for row_idx, (_, row) in enumerate(df.iterrows()):
            u_val = str(row[ctx_attr])
            if u_val not in ctx_vocab:
                continue  # unseen - оставляем 0.5

            u_id = ctx_vocab[u_val]

            proto = tuple(
                beh_vocabs[col].get(str(row[col]), -1)
                for col in beh_attrs
            )
            if -1 in proto:
                continue

            u_ids_list.append(u_id)
            proto_list.append(proto)
            valid_indices.append(row_idx)

        if not valid_indices:
            continue

        # Батчевый инференс
        BATCH = 4096
        all_scores = []

        with torch.no_grad():
            for start in range(0, len(valid_indices), BATCH):
                end = min(start + BATCH, len(valid_indices))

                u_tensor = torch.tensor(
                    u_ids_list[start:end], dtype=torch.long
                ).to(DEVICE)
                p_tensor = torch.tensor(
                    proto_list[start:end], dtype=torch.long
                ).to(DEVICE)

                f_scores = model.score(u_tensor, p_tensor)
                # Инвертируем: высокий f - легитимно - низкий скор фрода
                fraud_scores = 1.0 - torch.sigmoid(f_scores)
                all_scores.append(fraud_scores.cpu().numpy())

        all_scores = np.concatenate(all_scores)
        for i, row_idx in enumerate(valid_indices):
            scores_per_ctx[row_idx, ctx_idx] = all_scores[i]

    # Итоговый скор — среднее по контекстам
    return scores_per_ctx.mean(axis=1)


print("Вычисляем скоры на тестовой выборке...")
t0 = time.time()

ctkt_scores = score_transactions(
    df          = test_raw,
    cooc_data   = cooc_data,
    ctkt_models = ctkt_models,
    ctx_vocabs  = ctx_vocabs,
    beh_vocabs  = beh_vocabs,
    ctx_attrs   = CONTEXTUAL_ATTRS,
    beh_attrs   = BEHAVIORAL_ATTRS,
)

print(f"Время инференса (полный test): {time.time()-t0:.1f}s")
print(f"Скоры: min={ctkt_scores.min():.4f}, "
      f"max={ctkt_scores.max():.4f}, "
      f"mean={ctkt_scores.mean():.4f}")
print(f"Фрод в test: {test_raw['is_fraud'].sum():,} транзакций")

Вычисляем скоры на тестовой выборке...
Время инференса (полный test): 27.7s
Скоры: min=0.0288, max=0.8426, mean=0.2863
Фрод в test: 2,145 транзакций


In [10]:
def evaluate(
    model_name: str,
    y_true:     np.ndarray,
    scores:     np.ndarray,
    k_list:     list = [100, 500],
) -> dict:
    """
    Считает полный набор метрик.
    Используем ту же функцию что и в Notebook 2 для сопоставимости.
    """
    results = {"model": model_name}

    results["roc_auc"] = roc_auc_score(y_true, scores)
    results["pr_auc"]  = average_precision_score(y_true, scores)

    # Оптимальный порог по F1
    prec_curve, rec_curve, thresholds = precision_recall_curve(y_true, scores)
    f1_curve  = (2 * prec_curve * rec_curve
                 / (prec_curve + rec_curve + 1e-9))
    best_idx  = f1_curve.argmax()
    threshold = thresholds[best_idx]
    y_pred    = (scores >= threshold).astype(int)

    results["threshold"] = threshold
    results["precision"] = precision_score(y_true, y_pred, zero_division=0)
    results["recall"]    = recall_score(y_true, y_pred,    zero_division=0)
    results["f1"]        = f1_score(y_true, y_pred,        zero_division=0)

    # Метрики ранжирования
    n_fraud_total = y_true.sum()
    ranked_idx    = np.argsort(scores)[::-1]
    y_ranked      = y_true[ranked_idx]

    for k in k_list:
        top_k      = y_ranked[:k]
        fraud_in_k = top_k.sum()
        results[f"precision@{k}"] = fraud_in_k / k
        results[f"recall@{k}"]    = fraud_in_k / n_fraud_total
        results[f"ndcg@{k}"]      = ndcg_score(
            y_true.reshape(1, -1),
            scores.reshape(1, -1),
            k=k
        )

    return results


y_test = test_raw["is_fraud"].values

ctkt_results = evaluate("CTKT", y_test, ctkt_scores)

print("Метрики CTKT\n")
for k, v in ctkt_results.items():
    if k == "model":
        continue
    print(f"  {k:<20} {v:.4f}")

Метрики CTKT

  roc_auc              0.9470
  pr_auc               0.0753
  threshold            0.6664
  precision            0.1316
  recall               0.2191
  f1                   0.1645
  precision@100        0.0000
  recall@100           0.0000
  ndcg@100             0.0000
  precision@500        0.1140
  recall@500           0.0266
  ndcg@500             0.0991


In [11]:
print("Распределение скоров CTKT:")
print(f"  Перцентиль 99%:   {np.percentile(ctkt_scores, 99):.4f}")
print(f"  Перцентиль 99.9%: {np.percentile(ctkt_scores, 99.9):.4f}")
print(f"  Максимум:         {ctkt_scores.max():.4f}")

# Скоры фрода vs легитимных
fraud_scores  = ctkt_scores[y_test == 1]
legit_scores  = ctkt_scores[y_test == 0]
print(f"\n  Средний скор фрода:     {fraud_scores.mean():.4f}")
print(f"  Средний скор легитимных: {legit_scores.mean():.4f}")
print(f"  Топ-100 скоров содержит фрода: "
      f"{y_test[np.argsort(ctkt_scores)[::-1][:100]].sum()}")

Распределение скоров CTKT:
  Перцентиль 99%:   0.6436
  Перцентиль 99.9%: 0.7317
  Максимум:         0.8426

  Средний скор фрода:     0.5586
  Средний скор легитимных: 0.2853
  Топ-100 скоров содержит фрода: 0


In [12]:
def score_transactions_max(
    df:          pd.DataFrame,
    cooc_data:   dict,
    ctkt_models: dict,
    ctx_vocabs:  dict,
    beh_vocabs:  dict,
    ctx_attrs:   list,
    beh_attrs:   list,
) -> np.ndarray:
    """
    Версия с агрегацией max вместо mean.
    Транзакция считается подозрительной если хотя бы один
    контекст даёт высокий скор аномальности.
    """
    n = len(df)
    scores_per_ctx = np.full((n, len(ctx_attrs)), 0.5)

    for ctx_idx, ctx_attr in enumerate(ctx_attrs):
        model = ctkt_models[ctx_attr]
        model.eval()

        ctx_vocab     = ctx_vocabs[ctx_attr]
        u_ids_list    = []
        proto_list    = []
        valid_indices = []

        for row_idx, (_, row) in enumerate(df.iterrows()):
            u_val = str(row[ctx_attr])
            if u_val not in ctx_vocab:
                continue
            u_id  = ctx_vocab[u_val]
            proto = tuple(
                beh_vocabs[col].get(str(row[col]), -1)
                for col in beh_attrs
            )
            if -1 in proto:
                continue
            u_ids_list.append(u_id)
            proto_list.append(proto)
            valid_indices.append(row_idx)

        if not valid_indices:
            continue

        BATCH      = 4096
        all_scores = []

        with torch.no_grad():
            for start in range(0, len(valid_indices), BATCH):
                end      = min(start + BATCH, len(valid_indices))
                u_tensor = torch.tensor(
                    u_ids_list[start:end], dtype=torch.long
                ).to(DEVICE)
                p_tensor = torch.tensor(
                    proto_list[start:end], dtype=torch.long
                ).to(DEVICE)
                f_scores     = model.score(u_tensor, p_tensor)
                fraud_scores = 1.0 - torch.sigmoid(f_scores)
                all_scores.append(fraud_scores.cpu().numpy())

        all_scores = np.concatenate(all_scores)
        for i, row_idx in enumerate(valid_indices):
            scores_per_ctx[row_idx, ctx_idx] = all_scores[i]

    # max вместо mean
    return scores_per_ctx.max(axis=1)


print("Пересчитываем скоры с агрегацией max...")
t0 = time.time()
ctkt_scores_max = score_transactions_max(
    df          = test_raw,
    cooc_data   = cooc_data,
    ctkt_models = ctkt_models,
    ctx_vocabs  = ctx_vocabs,
    beh_vocabs  = beh_vocabs,
    ctx_attrs   = CONTEXTUAL_ATTRS,
    beh_attrs   = BEHAVIORAL_ATTRS,
)
print(f"Время: {time.time()-t0:.1f}s")

print(f"\nРаспределение скоров (max):")
print(f"  Перцентиль 99%:   {np.percentile(ctkt_scores_max, 99):.4f}")
print(f"  Перцентиль 99.9%: {np.percentile(ctkt_scores_max, 99.9):.4f}")
print(f"  Максимум:         {ctkt_scores_max.max():.4f}")

fraud_scores_max = ctkt_scores_max[y_test == 1]
legit_scores_max = ctkt_scores_max[y_test == 0]
print(f"\n  Средний скор фрода:      {fraud_scores_max.mean():.4f}")
print(f"  Средний скор легитимных: {legit_scores_max.mean():.4f}")
print(f"  Топ-100 содержит фрода:  "
      f"{y_test[np.argsort(ctkt_scores_max)[::-1][:100]].sum()}")

ctkt_results_max = evaluate("CTKT (max)", y_test, ctkt_scores_max)
print("\nМетрики CTKT (max агрегация)\n")
for k, v in ctkt_results_max.items():
    if k == "model":
        continue
    print(f"  {k:<20} {v:.4f}")

Пересчитываем скоры с агрегацией max...
Время: 27.5s

Распределение скоров (max):
  Перцентиль 99%:   0.7970
  Перцентиль 99.9%: 0.8672
  Максимум:         0.9493

  Средний скор фрода:      0.6838
  Средний скор легитимных: 0.4626
  Топ-100 содержит фрода:  26

Метрики CTKT (max агрегация)

  roc_auc              0.8940
  pr_auc               0.0681
  threshold            0.7858
  precision            0.0951
  recall               0.3044
  f1                   0.1449
  precision@100        0.2600
  recall@100           0.0121
  ndcg@100             0.2457
  precision@500        0.1660
  recall@500           0.0387
  ndcg@500             0.1739


### 3.7 Выбор стратегии агрегации скоров

CTKT строит отдельную модель для каждого контекстуального атрибута
$C \in \mathcal{A}_c$. Итоговый скор требует агрегации по контекстам.

Были протестированы две стратегии:

| Стратегия | ROC-AUC | PR-AUC | Precision@100 | NDCG@100 |
|---|---|---|---|---|
| Mean | **0.947** | **0.075** | 0.000 | 0.000 |
| Max | 0.894 | 0.068 | **0.260** | **0.246** |

$$s_{\text{mean}}(t) = \frac{1}{|\mathcal{A}_c|}
\sum_{C} \bigl(1 - \sigma(f_{u_C}(e))\bigr)$$

$$s_{\text{max}}(t) = \max_{C \in \mathcal{A}_c}
\bigl(1 - \sigma(f_{u_C}(e))\bigr)$$

In [13]:
# Валидационная проверка выбора стратегии агрегации (mean vs max)
# Отложенная валидация: последние 20% train по времени.
# Цель — подтвердить, что выбор max сделан без участия тестовой выборки.

train_sorted = train_raw.sort_values("unix_time").reset_index(drop=True)
split_v = int(len(train_sorted) * 0.8)
tr_v  = train_sorted.iloc[:split_v]
val_v = train_sorted.iloc[split_v:].reset_index(drop=True)

print(f"tr: {len(tr_v):,}  val: {len(val_v):,}  "
      f"фрод в val: {val_v['is_fraud'].mean():.4%}\n")

# Частоты и псевдо-рейтинги — только по tr_v
print("Строим cooccurrence по tr_v...")
cooc_v = {}
for c in CONTEXTUAL_ATTRS:
    t0 = time.time()
    cooc_v[c] = build_cooccurrence(tr_v, c, BEHAVIORAL_ATTRS,
                                   ctx_vocabs[c], beh_vocabs)
    print(f"  {c}: {len(cooc_v[c]):,} пар, {time.time()-t0:.1f}s")

# Обучение трёх контекстных моделей на tr_v
print("\nОбучаем CTKT на tr_v...\n")
models_v = {}
for c in CONTEXTUAL_ATTRS:
    print(f"── {c} ──")
    t0 = time.time()
    models_v[c] = train_ctkt(cooc_v, c, ctx_vocabs,
                             beh_vocabs, BEHAVIORAL_ATTRS)
    print(f"  Время: {time.time()-t0:.1f}s\n")

# Сравнение стратегий агрегации на val_v
y_v = val_v["is_fraud"].values
print("\nСтратегия | P@100 | P@500 | PR-AUC | ROC-AUC")
print("-" * 50)
for name, fn in [("mean", score_transactions),
                 ("max ", score_transactions_max)]:
    s = fn(val_v, cooc_v, models_v, ctx_vocabs, beh_vocabs,
           CONTEXTUAL_ATTRS, BEHAVIORAL_ATTRS)
    r = y_v[np.argsort(s)[::-1]]
    print(f"{name}      | {r[:100].mean():.3f} | {r[:500].mean():.3f} | "
          f"{average_precision_score(y_v, s):.4f} | "
          f"{roc_auc_score(y_v, s):.4f}")

tr: 1,037,340  val: 259,335  фрод в val: 0.5931%

Строим cooccurrence по tr_v...
  cc_num: 560,780 пар, 22.2s
  merchant: 202,343 пар, 19.6s
  state: 169,244 пар, 18.9s

Обучаем CTKT на tr_v...

── cc_num ──
  Preference events для cc_num: 2,026,280
    Epoch 1/5  loss=0.1711
    Epoch 2/5  loss=0.1569
    Epoch 3/5  loss=0.1561
    Epoch 4/5  loss=0.1552
    Epoch 5/5  loss=0.1545
  Время: 147.7s

── merchant ──
  Preference events для merchant: 337,370
    Epoch 1/5  loss=0.2008
    Epoch 2/5  loss=0.1101
    Epoch 3/5  loss=0.1043
    Epoch 4/5  loss=0.1033
    Epoch 5/5  loss=0.1031
  Время: 23.4s

── state ──
  Preference events для state: 4,304,364
    Epoch 1/5  loss=0.1121
    Epoch 2/5  loss=0.1049
    Epoch 3/5  loss=0.1047
    Epoch 4/5  loss=0.1045
    Epoch 5/5  loss=0.1044
  Время: 321.0s


Стратегия | P@100 | P@500 | PR-AUC | ROC-AUC
--------------------------------------------------
mean      | 0.090 | 0.238 | 0.1320 | 0.9497
max       | 0.250 | 0.254 | 0.1163 | 0.8977


Валидационная проверка на отложенных 20 % обучающей выборки воспроизводит ту же картину, что и на тесте: mean опережает max по глобальным метрикам (ROC-AUC 0,950 против 0,898; PR-AUC 0,132 против 0,116), тогда как max почти втрое превосходит mean по точности в верхней части очереди алертов (Precision@100 = 0,250 против 0,090). Таким образом, преимущество max в топе очереди является устойчивым свойством метода, а не особенностью тестовой выборки.

Агрегация через среднее сглаживает аномальные сигналы: транзакция может быть аномальной для держателя карты (`cc_num`), но нормальной для штата (`state`) — итоговый скор оказывается средним и не попадает в топ очереди алертов. Агрегация через максимум сохраняет сильнейший аномальный сигнал из любого контекста, что соответствует операционной логике антифрода: транзакция подозрительна, если она аномальна хотя бы в одном контексте.

Поскольку в постановке работы основными являются метрики ранжирования, в качестве финального варианта выбирается агрегация max — при осознанном проигрыше по PR-AUC.

In [14]:
# Финальные скоры CTKT — агрегация max
pd.DataFrame({
    "y_true": y_test,
    "score":  ctkt_scores_max,
}).to_csv(RESULTS / "scores_ctkt.csv", index=False)

print("Финальные скоры CTKT (max) сохранены: results/scores_ctkt.csv")
print(f"\nИтоговые метрики CTKT:")
for k, v in ctkt_results_max.items():
    if k == "model":
        continue
    print(f"  {k:<20} {v:.4f}")

Финальные скоры CTKT (max) сохранены: results/scores_ctkt.csv

Итоговые метрики CTKT:
  roc_auc              0.8940
  pr_auc               0.0681
  threshold            0.7858
  precision            0.0951
  recall               0.3044
  f1                   0.1449
  precision@100        0.2600
  recall@100           0.0121
  ndcg@100             0.2457
  precision@500        0.1660
  recall@500           0.0387
  ndcg@500             0.1739


## 4. GiPM3F — Gibbs iPM3F

### 4.1 Архитектура и ключевые идеи

GiPM3F (Yang et al., 2018) использует принципиально иной подход
по сравнению с CTKT. Вместо трансформации задачи фрода в
рекомендательную, метод обучает матричную факторизацию на
**нормальном поведении** и использует **ошибку предсказания**
как скоринговую функцию аномальности.

Интуиция: легитимный пользователь ведёт себя предсказуемо —
латентная модель хорошо восстанавливает его взаимодействия.
Мошенническая транзакция отклоняется от выученного профиля,
порождая высокую ошибку реконструкции.

**Матричная факторизация** (eq. IV.2, Yang et al. 2018):

$$\min_{U, V} \frac{1}{2}\left(\|U\|_F^2 + \|V\|_F^2\right)
+ C \sum_{(i,j) \in \mathcal{I}} h\!\left(Y_{ij} \cdot U_i V_j^\top\right)$$

где $U \in \mathbb{R}^{m \times k}$ — матрица латентных факторов
пользователей, $V \in \mathbb{R}^{n \times k}$ — матрица латентных
факторов объектов, $h(x) = \max(0, 1-x)$ — hinge loss,
$C$ — балансирующий коэффициент.

**Mean Prediction Error** (eq. IV.7):

$$\text{MPE}_u = \frac{\sum_{i \in \mathcal{I}} |p_{ui} - r_{ui}|}{|\mathcal{I}|}$$

где $p_{ui}$ — предсказанное значение, $r_{ui}$ — фактическое.
Высокий $\text{MPE}_u$ сигнализирует об аномальном поведении пользователя.

### 4.2 Адаптация к датасету Sparkov

Оригинальная статья разработана для рейтинговых систем
с целочисленными оценками $\{1, 2, 3, 4, 5\}$ и атаками
типа profile injection. Адаптация под карточный фрод:

| Аспект | Оригинал | Sparkov | Обоснование |
|---|---|---|---|
| Матрица | Пользователь × Товар, рейтинги 1–5 | Держатель × Мерчант, метки $\{0, 1\}$ | Прямая аналогия взаимодействий |
| Целевые значения | Минимальный рейтинг (nuke attack) | Мошеннические транзакции ($\texttt{is\_fraud}=1$) | Фрод аналогичен аномальным оценкам |
| Ранг $k$ | IBP (байесовский автовыбор) | Фиксированный, подбор на валидации | IBP требует MCMC — избыточно для бинарной задачи |
| MPE | Только по $r_{\min}$ или $r_{\max}$ | Только по фродовым транзакциям | Фокус на аномальных взаимодействиях |
| Скоринг | Порог по $\text{MPE}_i$ для items | $\text{MPE}_u$ как непрерывный скор | Нас интересует скор транзакции, не бинарная детекция |

**Что сохранено из оригинала без изменений:**
max-margin hinge loss (eq. IV.2), формула MPE (eq. IV.7),
логика фокуса только на аномальных взаимодействиях при
вычислении ошибки.

**Ключевое архитектурное решение:**
модель обучается на **всех** транзакциях train (легитимных и фродовых),
но MPE при инференсе вычисляется **только по фродовым** взаимодействиям
в матрице — это прямой аналог "только минимальный рейтинг" из статьи.

### 4.3 Построение матрицы взаимодействий

Строим разреженную матрицу $R \in \mathbb{R}^{m \times n}$,
где $m$ — число уникальных держателей карт,
$n$ — число уникальных мерчантов.

Элемент $R_{ij}$ определяется как:

$$R_{ij} = \begin{cases}
-1 & \text{если все транзакции держателя } i \text{ у мерчанта } j
\text{ легитимны} \\
+1 & \text{если хотя бы одна транзакция является мошеннической} \\
0  & \text{нет взаимодействий (отсутствующий элемент)}
\end{cases}$$

Знаковое кодирование $\{-1, +1\}$ совместимо с hinge loss:
$h(y \cdot \hat{y}) = \max(0, 1 - y \cdot \hat{y})$ достигает нуля
когда предсказание имеет правильный знак с достаточным отступом.

In [14]:
from scipy.sparse import csr_matrix, lil_matrix

print("Строим матрицу взаимодействий держатель × мерчант...")

# Словари идентификаторов — только по train
user_vocab = {
    u: i for i, u in enumerate(
        train_raw["cc_num"].astype(str).unique()
    )
}
merch_vocab = {
    m: i for i, m in enumerate(
        train_raw["merchant"].astype(str).unique()
    )
}

n_users  = len(user_vocab)
n_merch  = len(merch_vocab)
print(f"  Держателей карт: {n_users:,}")
print(f"  Мерчантов:       {n_merch:,}")

# Строим матрицу: +1 если есть фрод, -1 если только легитимные
R = lil_matrix((n_users, n_merch), dtype=np.float32)

for _, row in train_raw.iterrows():
    u = user_vocab.get(str(row["cc_num"]))
    m = merch_vocab.get(str(row["merchant"]))
    if u is None or m is None:
        continue

    label = int(row["is_fraud"])
    if label == 1:
        R[u, m] = 1.0   # фрод - +1
    elif R[u, m] == 0:
        R[u, m] = -1.0  # легитимная, если ещё не помечена как фрод

R_csr = R.tocsr()

n_observed = R_csr.nnz
n_fraud_entries = (R_csr.data == 1.0).sum()
print(f"\n  Наблюдаемых элементов: {n_observed:,}")
print(f"  Из них фрод (+1):      {n_fraud_entries:,} "
      f"({n_fraud_entries/n_observed:.2%})")
print(f"  Разреженность матрицы: "
      f"{1 - n_observed/(n_users*n_merch):.4%}")

Строим матрицу взаимодействий держатель × мерчант...
  Держателей карт: 983
  Мерчантов:       693

  Наблюдаемых элементов: 479,072
  Из них фрод (+1):      7,391 (1.54%)
  Разреженность матрицы: 29.6743%


### 4.4 Реализация max-margin матричной факторизации

Оптимизируем задачу (eq. IV.2) через SGD с hinge loss.
Предсказание для пары $(u, i)$:

$$\hat{r}_{ui} = U_i \cdot V_j^\top$$

Функция потерь на батче:

$$\mathcal{L} = \frac{1}{2}\left(\|U\|_F^2 + \|V\|_F^2\right)
+ C \cdot \frac{1}{|\mathcal{B}|}
\sum_{(i,j) \in \mathcal{B}} \max\!\left(0,\ 1 - Y_{ij} \cdot \hat{r}_{ij}\right)$$

Размерность латентного пространства $k$ подбирается на валидационной
выборке по метрике PR-AUC вместо байесовского IBP из оригинала.

### Примечание об IBP

Оригинальная реализация GiPM3F использует байесовский
непараметрический подход на основе Indian Buffet Process (IBP)
через MCMC-сэмплирование (Gibbs sampling) для автоматического
определения ранга $k$. Полная реализация IBP выходит за рамки
текущей работы ввиду значительной вычислительной сложности
алгоритма Гиббса и зависимости от дополнительной библиотеки
авторов.

В адаптированной постановке ранг $k$ подбирается на валидационной
выборке по метрике PR-AUC из набора кандидатов
$k \in \{8, 16, 32, 64, 128\}$. Это стандартная практика при
воспроизведении методов с байесовскими компонентами в условиях
ограниченных вычислительных ресурсов.

In [17]:
class GiPM3FModel(nn.Module):
    """
    Max-margin матричная факторизация для детекции аномалий.
    Адаптация GiPM3F (Yang et al., 2018) под бинарный карточный фрод.
    Предсказание: r_hat = U_i · V_j
    Потеря: взвешенный hinge loss + L2 регуляризация (eq. IV.2)
    """
    def __init__(self, n_users: int, n_items: int, k: int = 32):
        super().__init__()
        self.U = nn.Embedding(n_users, k)
        self.V = nn.Embedding(n_items, k)
        nn.init.normal_(self.U.weight, mean=0.0, std=0.01)
        nn.init.normal_(self.V.weight, mean=0.0, std=0.01)

    def forward(
        self,
        user_ids:   torch.Tensor,
        item_ids:   torch.Tensor,
        labels:     torch.Tensor,
        C:          float = 1.0,
        pos_weight: float = 172.0,
    ) -> torch.Tensor:
        u     = self.U(user_ids)
        v     = self.V(item_ids)
        r_hat = (u * v).sum(dim=1)

        hinge   = torch.clamp(1.0 - labels * r_hat, min=0.0)
        weights = torch.where(
            labels > 0,
            torch.full_like(labels, pos_weight),
            torch.ones_like(labels)
        )
        hinge = (hinge * weights).mean()

        reg = 0.5 * (u.pow(2).sum() + v.pow(2).sum()) / user_ids.size(0)
        return reg + C * hinge

    def predict(
        self,
        user_ids: torch.Tensor,
        item_ids: torch.Tensor,
    ) -> torch.Tensor:
        u = self.U(user_ids)
        v = self.V(item_ids)
        return (u * v).sum(dim=1)

In [19]:
from sklearn.model_selection import train_test_split

# Разбиваем train на train/val по времени (80/20)
train_size = int(len(train_raw) * 0.8)
tr_df  = train_raw.iloc[:train_size]
val_df = train_raw.iloc[train_size:]

print(f"Train для подбора k: {len(tr_df):,}")
print(f"Val для подбора k:   {len(val_df):,}")


def train_gipm3f(
    df:          pd.DataFrame,
    n_users:     int,
    n_merch:     int,
    user_vocab:  dict,
    merch_vocab: dict,
    k:           int   = 32,
    n_epochs:    int   = 5,
    batch_size:  int   = 4096,
    lr:          float = 0.001,
    C:           float = 1.0,
) -> GiPM3FModel:
    """Обучает GiPM3F на переданном датафрейме."""

    users, items, labels = [], [], []
    for _, row in df.iterrows():
        u = user_vocab.get(str(row["cc_num"]))
        m = merch_vocab.get(str(row["merchant"]))
        if u is None or m is None:
            continue
        users.append(u)
        items.append(m)
        labels.append(1.0 if row["is_fraud"] == 1 else -1.0)

    users  = torch.tensor(users,  dtype=torch.long)
    items  = torch.tensor(items,  dtype=torch.long)
    labels = torch.tensor(labels, dtype=torch.float32)

    dataset    = torch.utils.data.TensorDataset(users, items, labels)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    model     = GiPM3FModel(n_users, n_merch, k=k).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    model.train()
    for epoch in range(n_epochs):
        total_loss = 0.0
        for u_b, i_b, l_b in dataloader:
            u_b, i_b, l_b = u_b.to(DEVICE), i_b.to(DEVICE), l_b.to(DEVICE)
            optimizer.zero_grad()
            loss = model(u_b, i_b, l_b, C=C)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"    k={k} | Epoch {epoch+1}/{n_epochs} "
              f"loss={total_loss/len(dataloader):.4f}")

    return model


def compute_mpe_scores(
    df:          pd.DataFrame,
    model:       GiPM3FModel,
    user_vocab:  dict,
    merch_vocab: dict,
) -> np.ndarray:
    """
    Вычисляет MPE_u для каждой транзакции в df.
    MPE считается только по фродовым транзакциям —
    аналог 'только r_min' из оригинальной статьи.
    """
    fraud_df    = df[df["is_fraud"] == 1]
    user_errors = defaultdict(list)

    model.eval()
    BATCH = 4096

    fraud_users = []
    fraud_merch = []

    for _, row in fraud_df.iterrows():
        u = user_vocab.get(str(row["cc_num"]))
        m = merch_vocab.get(str(row["merchant"]))
        if u is not None and m is not None:
            fraud_users.append(u)
            fraud_merch.append(m)

    if fraud_users:
        with torch.no_grad():
            for start in range(0, len(fraud_users), BATCH):
                end   = min(start + BATCH, len(fraud_users))
                u_t   = torch.tensor(
                    fraud_users[start:end], dtype=torch.long
                ).to(DEVICE)
                m_t   = torch.tensor(
                    fraud_merch[start:end], dtype=torch.long
                ).to(DEVICE)
                preds = model.predict(u_t, m_t).cpu().numpy()

                for u_id, pred in zip(fraud_users[start:end], preds):
                    user_errors[u_id].append(abs(pred - 1.0))

    mpe_per_user = {
        u: np.mean(errs) for u, errs in user_errors.items()
    }
    global_mpe = np.mean(list(mpe_per_user.values())) if mpe_per_user else 0.5

    scores = np.full(len(df), global_mpe)
    for row_idx, (_, row) in enumerate(df.iterrows()):
        u = user_vocab.get(str(row["cc_num"]))
        if u is not None and u in mpe_per_user:
            scores[row_idx] = mpe_per_user[u]

    return scores


# Подбираем k на валидации
K_CANDIDATES = [8, 16, 32, 64, 128]
best_k, best_prauc = None, -1

print("Подбор ранга k (расширенный поиск, 10 эпох)...\n")
for k in K_CANDIDATES:
    print(f"  k={k}:")
    model_k = train_gipm3f(
        tr_df, n_users, n_merch, user_vocab, merch_vocab,
        k=k, n_epochs=10, lr=0.001
    )
    val_scores = compute_mpe_scores(
        val_df, model_k, user_vocab, merch_vocab
    )
    pr_auc_k = average_precision_score(
        val_df["is_fraud"].values, val_scores
    )
    print(f"    PR-AUC на val: {pr_auc_k:.4f}\n")

    if pr_auc_k > best_prauc:
        best_prauc = pr_auc_k
        best_k     = k

print(f"Лучший k = {best_k} (PR-AUC = {best_prauc:.4f})")

Train для подбора k: 1,037,340
Val для подбора k:   259,335
Подбор ранга k (расширенный поиск, 10 эпох)...

  k=8:
    k=8 | Epoch 1/10 loss=1.9839
    k=8 | Epoch 2/10 loss=1.9858
    k=8 | Epoch 3/10 loss=1.9792
    k=8 | Epoch 4/10 loss=1.9657
    k=8 | Epoch 5/10 loss=1.9449
    k=8 | Epoch 6/10 loss=1.9168
    k=8 | Epoch 7/10 loss=1.8851
    k=8 | Epoch 8/10 loss=1.8544
    k=8 | Epoch 9/10 loss=1.8328
    k=8 | Epoch 10/10 loss=1.8175
    PR-AUC на val: 0.0224

  k=16:
    k=16 | Epoch 1/10 loss=1.9844
    k=16 | Epoch 2/10 loss=1.9827
    k=16 | Epoch 3/10 loss=1.9776
    k=16 | Epoch 4/10 loss=1.9550
    k=16 | Epoch 5/10 loss=1.9165
    k=16 | Epoch 6/10 loss=1.8711
    k=16 | Epoch 7/10 loss=1.8362
    k=16 | Epoch 8/10 loss=1.8163
    k=16 | Epoch 9/10 loss=1.8011
    k=16 | Epoch 10/10 loss=1.7927
    PR-AUC на val: 0.0225

  k=32:
    k=32 | Epoch 1/10 loss=1.9817
    k=32 | Epoch 2/10 loss=1.9825
    k=32 | Epoch 3/10 loss=1.9629
    k=32 | Epoch 4/10 loss=1.9116
    k=3

### 4.5 Результаты подбора ранга $k$

| $k$ | PR-AUC (val) |
|---|---|
| 8 | 0.0224 |
| 16 | 0.0225 |
| 32 | 0.0227 |
| 64 | **0.0228** |
| 128 | 0.0228 |

PR-AUC практически не изменяется при увеличении $k$ — плато
достигается уже при $k=32$. Это говорит о том, что ограничение
качества определяется не размерностью латентного пространства,
а фундаментальным несоответствием между доменом применения
метода и задачей карточного фрода.

GiPM3F разработан для обнаружения profile injection атак
в рейтинговых системах, где аномалия проявляется как
систематическое отклонение в распределении оценок.
В карточном фроде аномалия — это единичная нетипичная
транзакция, что структурно отличается от паттернов
для которых метод был спроектирован.

Выбираем $k = 64$ как лучший по валидации.

In [20]:
print(f"Обучаем финальную модель GiPM3F с k={best_k} на полном train...\n")

t0 = time.time()
gipm3f_model = train_gipm3f(
    train_raw, n_users, n_merch, user_vocab, merch_vocab,
    k=best_k, n_epochs=10, lr=0.001
)
print(f"\nВремя обучения: {time.time()-t0:.1f}s")

# Скоры на тесте
print("\nВычисляем скоры на тестовой выборке...")
t0 = time.time()
gipm3f_scores = compute_mpe_scores(
    test_raw, gipm3f_model, user_vocab, merch_vocab
)
print(f"Время инференса (полный test): {time.time()-t0:.1f}s")

print(f"\nРаспределение скоров:")
print(f"  min={gipm3f_scores.min():.4f}  "
      f"max={gipm3f_scores.max():.4f}  "
      f"mean={gipm3f_scores.mean():.4f}")

fraud_s = gipm3f_scores[y_test == 1]
legit_s = gipm3f_scores[y_test == 0]
print(f"  Средний скор фрода:      {fraud_s.mean():.4f}")
print(f"  Средний скор легитимных: {legit_s.mean():.4f}")

# Метрики
gipm3f_results = evaluate("GiPM3F", y_test, gipm3f_scores)
for k, v in gipm3f_results.items():
    if k == "model":
        continue
    print(f"  {k:<20} {v:.4f}")

# Сохраняем скоры
pd.DataFrame({
    "y_true": y_test,
    "score":  gipm3f_scores,
}).to_csv(RESULTS / "scores_gipm3f.csv", index=False)
print("\nСкоры GiPM3F сохранены: results/scores_gipm3f.csv")

Обучаем финальную модель GiPM3F с k=64 на полном train...

    k=64 | Epoch 1/10 loss=1.9911
    k=64 | Epoch 2/10 loss=1.9880
    k=64 | Epoch 3/10 loss=1.9580
    k=64 | Epoch 4/10 loss=1.9028
    k=64 | Epoch 5/10 loss=1.8725
    k=64 | Epoch 6/10 loss=1.8587
    k=64 | Epoch 7/10 loss=1.8537
    k=64 | Epoch 8/10 loss=1.8491
    k=64 | Epoch 9/10 loss=1.8473
    k=64 | Epoch 10/10 loss=1.8462

Время обучения: 134.3s

Вычисляем скоры на тестовой выборке...
Время инференса (полный test): 8.1s

Распределение скоров:
  min=0.4453  max=1.2817  mean=1.1282
  Средний скор фрода:      1.1328
  Средний скор легитимных: 1.1282
  roc_auc              0.5357
  pr_auc               0.0115
  threshold            1.2334
  precision            0.0372
  recall               0.0382
  f1                   0.0377
  precision@100        0.0200
  recall@100           0.0009
  ndcg@100             0.0335
  precision@500        0.0400
  recall@500           0.0093
  ndcg@500             0.0394

Скоры GiPM

## 5. Итоги Notebook 3

### Сравнительная таблица результатов

| Метрика | CTKT (max) | GiPM3F |
|---|---|---|
| ROC-AUC | **0.894** | 0.536 |
| PR-AUC | **0.068** | 0.012 |
| F1 | **0.145** | 0.038 |
| Precision@100 | **0.260** | 0.020 |
| Recall@100 | **0.012** | 0.001 |
| NDCG@100 | **0.246** | 0.034 |
| Precision@500 | **0.166** | 0.040 |
| NDCG@500 | **0.174** | 0.039 |

### Анализ результатов

**CTKT** демонстрирует умеренное качество: ROC-AUC 0.894
сопоставим с LightGBM (0.903), однако PR-AUC (0.068) значительно
уступает бейзлайну (0.546). Метрики ранжирования показывают
что метод способен выводить реальный фрод в топ очереди алертов
(Precision@100 = 0.26), хотя и значительно хуже LightGBM (1.0).

Агрегация скоров через $\max$ по контекстам оказалась
принципиально важной: агрегация через среднее давала
Precision@100 = 0.0, тогда как $\max$ — 0.26. Это отражает
природу задачи: транзакция подозрительна, если она аномальна
**хотя бы в одном** из контекстов (держатель, мерчант, штат).

**GiPM3F** показывает результаты близкие к случайному
классификатору (ROC-AUC 0.536, разрыв средних скоров
фрода и легитимных < 0.005). Причина — фундаментальное
несоответствие домена: метод разработан для обнаружения
profile injection атак в рейтинговых системах, где аномалия
проявляется как систематическое отклонение в распределении
оценок конкретного товара. В карточном фроде аномалия —
это единичная нетипичная транзакция конкретного держателя,
что структурно отличается.

Дополнительным ограничением является замена байесовского
IBP фиксированным подбором ранга $k$: PR-AUC не улучшается
при росте $k$ от 8 до 128, что подтверждает — проблема
не в размерности, а в самой постановке задачи.

### Выводы

Из двух рассмотренных методов только CTKT демонстрирует
осмысленный сигнал детекции фрода. GiPM3F в адаптированной
постановке не справляется с задачей — что соответствует
теоретическому анализу из раздела 3.1 курсовой работы,
где указывалось на структурное несоответствие метода
и домена карточного фрода.

Эти результаты закладывают базу для сравнения с графовыми
и последовательными методами в Notebooks 4–5.